# Shapes of Stories: Curated Demo

Canonical books processed through GPT-2 medium.
Overlapping chunks to eliminate boundary artifacts.
Passage-level heuristics to label prose modes.
Exports JSON for interactive visualisation.

In [ ]:
!pip install -q transformers datasets umap-learn scikit-learn matplotlib pandas scipy 2>/dev/null

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, matplotlib.pyplot as plt, pandas as pd
import json, gc, os, warnings, random, re
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE = '/content/story_demo'
os.makedirs(SAVE, exist_ok=True)
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print(f'Device: {device}')

## 1. Curated Corpus

Recognisable books from Project Gutenberg. We search by title
and take a 10,000-token segment from the middle of each.

In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Books we want — title substrings to match against pg19
WANTED = [
    'Pride and Prejudice',
    'Frankenstein',
    'Dracula',
    'A Christmas Carol',
    'Adventures of Sherlock Holmes',
    'Heart of Darkness',
    'War of the Worlds',
    'Alice\'s Adventures in Wonderland',
    'Moby Dick',
    'Picture of Dorian Gray',
    'Jane Eyre',
    'Wuthering Heights',
    'Great Expectations',
    'Tale of Two Cities',
    'The Odyssey',
    'Crime and Punishment',
    'Anna Karenina',
    'Treasure Island',
    'The Jungle Book',
    'The Time Machine',
    'The Strange Case of Dr. Jekyll',
    'The Importance of Being Earnest',
    'The Turn of the Screw',
    'Sense and Sensibility',
    'Little Women',
    'The Scarlet Letter',
    'The Call of the Wild',
    'Twenty Thousand Leagues',
    'Les Misérables',
    'Don Quixote',
]

SEG_LEN = 10000
MIN_BOOK_LEN = 15000

wanted_lower = [w.lower() for w in WANTED]
found = {}  # title -> book data

print(f'Searching pg19 for {len(WANTED)} books...')
pg = load_dataset('emozilla/pg19', split='train', streaming=True)

for ex in pg:
    title = ex.get('short_book_title', ex.get('title', ''))
    title_lower = title.lower()
    
    for wi, wanted in enumerate(wanted_lower):
        if wanted in title_lower and WANTED[wi] not in found:
            text = ex['text']
            toks = tokenizer.encode(text)
            if len(toks) < MIN_BOOK_LEN:
                continue
            
            mid = len(toks) // 2
            start = mid - SEG_LEN // 2
            seg_toks = toks[start:start + SEG_LEN]
            
            found[WANTED[wi]] = {
                'title': title,
                'search_key': WANTED[wi],
                'tokens': seg_toks,
                'full_len': len(toks),
                'seg_text': tokenizer.decode(seg_toks)
            }
            print(f'  Found: {title} ({len(toks)} tokens)')
            break
    
    if len(found) >= len(WANTED):
        break

# Also grab some extras to fill out if we didn't find all
if len(found) < 15:
    print(f'\nOnly found {len(found)}, grabbing additional books...')
    extra_count = 0
    pg2 = load_dataset('emozilla/pg19', split='train', streaming=True)
    for ex in pg2:
        title = ex.get('short_book_title', ex.get('title', ''))
        if title in [v['title'] for v in found.values()]:
            continue
        text = ex['text']
        toks = tokenizer.encode(text)
        if len(toks) < MIN_BOOK_LEN:
            continue
        mid = len(toks) // 2
        seg_toks = toks[mid - SEG_LEN//2 : mid + SEG_LEN//2]
        found[f'extra_{extra_count}'] = {
            'title': title, 'search_key': title,
            'tokens': seg_toks, 'full_len': len(toks),
            'seg_text': tokenizer.decode(seg_toks)
        }
        extra_count += 1
        print(f'  Extra: {title}')
        if len(found) >= 20:
            break

books = list(found.values())
print(f'\n{len(books)} books ready')
for b in books:
    print(f'  {b["title"]} ({b["full_len"]} tokens)')

## 2. Collect Hidden States (Overlapping Chunks)

Process in overlapping 1024-token chunks with 512 overlap.
Keep only hidden states from the second half of each chunk
(where the model has full context). This eliminates the
chunk boundary artifact.

In [ ]:
from transformers import GPT2LMHeadModel

MODEL_NAME = 'gpt2-medium'
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device); model.eval()
config = model.config
D_MODEL = config.n_embd
N_LAYERS = config.n_layer
CTX = config.n_positions
LAYER = N_LAYERS - 1  # penultimate
OVERLAP = 512  # overlap between chunks
STRIDE = CTX - OVERLAP  # 512 new tokens per chunk

print(f'Model: {MODEL_NAME}, d={D_MODEL}, layers={N_LAYERS}')
print(f'Chunks: {CTX} tokens, stride {STRIDE}, overlap {OVERLAP}')
print(f'Collecting layer {LAYER}')

In [ ]:
all_hidden = []   # per book: (n_tokens, D_MODEL)
all_entropy = []  # per book: (n_tokens,)
all_tokens = []   # per book: list of token ids (for text reconstruction)

print(f'Processing {len(books)} books with overlapping chunks...')
with torch.no_grad():
    for bi, book in enumerate(books):
        toks = book['tokens']
        book_h = []
        book_ent = []
        book_tok_ids = []
        
        chunk_starts = list(range(0, len(toks) - CTX + 1, STRIDE))
        if not chunk_starts:
            chunk_starts = [0]
        
        for ci, cs in enumerate(chunk_starts):
            chunk = toks[cs:cs+CTX]
            ids = torch.tensor([chunk], device=device)
            out = model(ids, output_hidden_states=True)
            
            h = out.hidden_states[LAYER][0].cpu()
            lp = F.log_softmax(out.logits[0], dim=-1)
            ent = -(torch.exp(lp) * lp).sum(dim=-1).cpu().numpy()
            
            if ci == 0:
                # First chunk: keep everything
                book_h.append(h)
                book_ent.append(ent)
                book_tok_ids.extend(chunk)
            else:
                # Subsequent chunks: keep only the non-overlapping part
                book_h.append(h[OVERLAP:])
                book_ent.append(ent[OVERLAP:])
                book_tok_ids.extend(chunk[OVERLAP:])
            
            del out, lp
        
        all_hidden.append(torch.cat(book_h, dim=0))
        all_entropy.append(np.concatenate(book_ent))
        all_tokens.append(book_tok_ids)
        print(f'  {bi+1}. {book["title"][:40]}: {all_hidden[-1].shape[0]} tokens')

del model; torch.cuda.empty_cache(); gc.collect()
print(f'\nDone. {sum(h.shape[0] for h in all_hidden)} total tokens.')

## 3. Window Pooling + Passage Features

W=64 tokens per passage. For each passage, compute:
- Mean hidden state (768-d)
- Text snippet
- Prose mode heuristics: quote density, mean sentence length, entropy

In [ ]:
WINDOW = 64

passage_vecs = []
passage_meta = []
story_ranges = []

for bi, h in enumerate(all_hidden):
    n_windows = h.shape[0] // WINDOW
    if n_windows < 4: continue
    
    start = len(passage_vecs)
    for wi in range(n_windows):
        s_tok = wi * WINDOW
        e_tok = (wi + 1) * WINDOW
        
        # Hidden state
        passage_vecs.append(h[s_tok:e_tok].mean(dim=0).numpy())
        
        # Decode passage text
        tok_ids = all_tokens[bi][s_tok:e_tok]
        passage_text = tokenizer.decode(tok_ids)
        
        # Prose mode heuristics
        n_quotes = passage_text.count('"') + passage_text.count("'") + passage_text.count('``') + passage_text.count("''")
        quote_density = n_quotes / max(len(passage_text), 1)
        
        # Sentence length proxy: count periods, exclamations, question marks
        n_sentences = len(re.findall(r'[.!?]+', passage_text))
        mean_sent_len = len(passage_text.split()) / max(n_sentences, 1)
        
        # Entropy
        mean_ent = all_entropy[bi][s_tok:e_tok].mean()
        
        # Dialogue fraction: rough estimate from quote marks
        in_quotes = 0
        inside = False
        for ch in passage_text:
            if ch == '"':
                inside = not inside
            elif inside:
                in_quotes += 1
        dialogue_frac = in_quotes / max(len(passage_text), 1)
        
        passage_meta.append({
            'book': bi,
            'title': books[bi]['title'],
            'window': wi,
            'pos_frac': (wi + 0.5) / n_windows,
            'text_snippet': passage_text[:150].replace('\n', ' '),
            'full_text': passage_text.replace('\n', ' '),
            'quote_density': quote_density,
            'dialogue_frac': dialogue_frac,
            'mean_sent_len': mean_sent_len,
            'entropy': float(mean_ent)
        })
    
    story_ranges.append((start, len(passage_vecs)))

passage_vecs = np.stack(passage_vecs)
n_books = len(story_ranges)
print(f'{n_books} books, {len(passage_vecs)} passages')

## 4. Autoencoder

In [ ]:
from sklearn.decomposition import IncrementalPCA, PCA

# PCA pre-reduction
print('PCA 1024 → 128...')
pca_pre = IncrementalPCA(n_components=128, batch_size=2000)
for i in range(0, len(passage_vecs), 2000): pca_pre.partial_fit(passage_vecs[i:i+2000])
pv_reduced = np.empty((len(passage_vecs), 128), dtype=np.float32)
for i in range(0, len(passage_vecs), 2000):
    chunk = passage_vecs[i:i+2000]
    pv_reduced[i:i+len(chunk)] = pca_pre.transform(chunk)

ev = pca_pre.explained_variance_ratio_
cs = np.cumsum(ev)
print(f'95% at {int(np.searchsorted(cs, 0.95))+1}, 99% at {int(np.searchsorted(cs, 0.99))+1}')

In [ ]:
class AE(nn.Module):
    def __init__(s, ind, hid, lat):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(ind, hid), nn.ReLU(),
                              nn.Linear(hid, hid//2), nn.ReLU(),
                              nn.Linear(hid//2, lat))
        s.dec = nn.Sequential(nn.Linear(lat, hid//2), nn.ReLU(),
                              nn.Linear(hid//2, hid), nn.ReLU(),
                              nn.Linear(hid, ind))
    def encode(s, x): return s.enc(x)
    def forward(s, x): z = s.enc(x); return s.dec(z), z

def train_ae(data, k, ind=128, hid=256, ep=80, bs=256, lr=1e-3):
    data_t = torch.tensor(data, dtype=torch.float32)
    n = len(data_t); perm = torch.randperm(n); nt = int(0.85*n)
    tr, te = data_t[perm[:nt]], data_t[perm[nt:]]
    m = AE(ind, hid, k).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, ep)
    ld = DataLoader(TensorDataset(tr), batch_size=bs, shuffle=True)
    for e in range(ep):
        m.train()
        for (b,) in ld:
            b = b.to(device); r, z = m(b)
            l = F.mse_loss(r, b); opt.zero_grad(); l.backward(); opt.step()
        sch.step()
    m.eval()
    with torch.no_grad():
        rt, _ = m(tr[:1000].to(device)); ft = F.mse_loss(rt, tr[:1000].to(device)).item()
        re2, _ = m(te.to(device)); fte = F.mse_loss(re2, te.to(device)).item()
    return m, ft, fte

sweep = {}
for k in [4, 8, 16]:
    print(f'k={k}...', end=' ')
    m, ft, fte = train_ae(pv_reduced, k)
    r = fte/ft if ft > 1e-15 else float('inf')
    sweep[k] = {'ft': ft, 'fte': fte, 'model': m}
    print(f'train={ft:.3e} test={fte:.3e} ratio={r:.2f}x')

In [ ]:
K = 8  # ← adjust
ae = sweep[K]['model']; ae.eval()

with torch.no_grad():
    pv_t = torch.tensor(pv_reduced, dtype=torch.float32)
    latents = torch.cat([ae.encode(pv_t[i:i+500].to(device)).cpu()
                         for i in range(0, len(pv_t), 500)]).numpy()
print(f'Latent: {latents.shape}')

## 5. UMAP + Visualisations

In [ ]:
import umap

print('UMAP...')
umap_emb = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42).fit_transform(latents)

# Store coordinates in metadata
for i, m in enumerate(passage_meta):
    m['umap_x'] = float(umap_emb[i, 0])
    m['umap_y'] = float(umap_emb[i, 1])
    m['latent'] = latents[i].tolist()

In [ ]:
# Colour palette for books
book_titles = [books[passage_meta[s]['book']]['title'] for s, e in story_ranges]
n_colors = len(story_ranges)
if n_colors <= 10:
    cmap = plt.cm.tab10
elif n_colors <= 20:
    cmap = plt.cm.tab20
else:
    cmap = plt.cm.gist_ncar
colors = [cmap(i / n_colors) for i in range(n_colors)]

In [ ]:
# Main trajectory plot
fig, ax = plt.subplots(figsize=(18, 16))
ax.scatter(umap_emb[:,0], umap_emb[:,1], c='lightgrey', s=1, alpha=0.08)

from matplotlib.lines import Line2D
legend_elements = []

for bi, (s, e) in enumerate(story_ranges):
    traj = umap_emb[s:e]
    if len(traj) < 2: continue
    title = books[passage_meta[s]['book']]['title']
    short_title = title[:35]
    
    ax.plot(traj[:,0], traj[:,1], '-', color=colors[bi], alpha=0.45, linewidth=0.8)
    sizes = np.linspace(6, 25, len(traj))
    ax.scatter(traj[:,0], traj[:,1], c=[colors[bi]], s=sizes, alpha=0.5, zorder=5)
    ax.scatter(traj[0,0], traj[0,1], c=[colors[bi]], s=70, marker='o',
               edgecolors='black', linewidths=1.2, zorder=10)
    ax.scatter(traj[-1,0], traj[-1,1], c=[colors[bi]], s=70, marker='s',
               edgecolors='black', linewidths=1.2, zorder=10)
    
    legend_elements.append(Line2D([0],[0], color=colors[bi], linewidth=2, label=short_title))

legend_elements += [
    Line2D([0],[0], marker='o', color='grey', markeredgecolor='black', markersize=8, linestyle='None', label='Start'),
    Line2D([0],[0], marker='s', color='grey', markeredgecolor='black', markersize=8, linestyle='None', label='End')
]
ax.legend(handles=legend_elements, fontsize=8, loc='upper right', framealpha=0.9)
ax.set_title('Shapes of Stories: Book Trajectories Through Narrative Space', fontsize=15)
ax.set_aspect('equal')
plt.tight_layout(); plt.savefig(f'{SAVE}/trajectories_curated.png', dpi=200); plt.show()

In [ ]:
# Individual book panels — 4-5 per row
n_per_row = 5
n_rows = (n_books + n_per_row - 1) // n_per_row
fig, axes = plt.subplots(n_rows, n_per_row, figsize=(5*n_per_row, 5*n_rows))
axes = axes.flat if n_rows > 1 else [axes] if n_books == 1 else axes

for bi, (s, e) in enumerate(story_ranges):
    ax = axes[bi]
    ax.scatter(umap_emb[:,0], umap_emb[:,1], c='lightgrey', s=0.3, alpha=0.05)
    traj = umap_emb[s:e]
    ax.plot(traj[:,0], traj[:,1], '-', color=colors[bi], alpha=0.5, linewidth=0.8)
    ax.scatter(traj[:,0], traj[:,1], c=[colors[bi]], s=8, alpha=0.6)
    ax.scatter(traj[0,0], traj[0,1], c=[colors[bi]], s=50, marker='o', edgecolors='black', linewidths=1)
    ax.scatter(traj[-1,0], traj[-1,1], c=[colors[bi]], s=50, marker='s', edgecolors='black', linewidths=1)
    title = books[passage_meta[s]['book']]['title'][:30]
    ax.set_title(title, fontsize=9, color=colors[bi])
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])

for ax in list(axes)[n_books:]: ax.set_visible(False)
plt.suptitle('Individual Book Trajectories', fontsize=14)
plt.tight_layout(); plt.savefig(f'{SAVE}/individual_trajectories.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Prose Mode Analysis

Test whether UMAP position correlates with prose mode heuristics.

In [ ]:
# Colour by prose features
dialogue_fracs = np.array([m['dialogue_frac'] for m in passage_meta])
entropies = np.array([m['entropy'] for m in passage_meta])
sent_lens = np.array([m['mean_sent_len'] for m in passage_meta])
pos_fracs = np.array([m['pos_frac'] for m in passage_meta])

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, data, cmap_name, title in zip(axes.flat, 
    [dialogue_fracs, entropies, sent_lens, pos_fracs],
    ['YlOrRd', 'plasma', 'viridis', 'coolwarm'],
    ['Dialogue Fraction', 'Model Entropy', 'Mean Sentence Length', 'Position in Book']):
    sc = ax.scatter(umap_emb[:,0], umap_emb[:,1], c=data, cmap=cmap_name, s=3, alpha=0.4)
    plt.colorbar(sc, ax=ax, shrink=0.7)
    ax.set_title(title, fontsize=12); ax.set_aspect('equal')

plt.suptitle('What Does the Narrative Space Encode?', fontsize=14, y=1.02)
plt.tight_layout(); plt.savefig(f'{SAVE}/prose_features.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Quantitative: correlation of UMAP coords with features
from scipy.stats import spearmanr

print('Spearman correlations with UMAP position:')
print(f'{"Feature":<25} {"UMAP-x":>10} {"UMAP-y":>10}')
print('-' * 45)
for name, data in [('Dialogue fraction', dialogue_fracs),
                    ('Entropy', entropies),
                    ('Sentence length', sent_lens),
                    ('Position in book', pos_fracs)]:
    rx, _ = spearmanr(umap_emb[:,0], data)
    ry, _ = spearmanr(umap_emb[:,1], data)
    print(f'{name:<25} {rx:>10.3f} {ry:>10.3f}')

# Also correlate with latent axes
print(f'\nSpearman correlations with AE latent axes:')
print(f'{"Feature":<25}', ' '.join([f'{"Ax"+str(i):>8}' for i in range(K)]))
print('-' * (25 + 9*K))
for name, data in [('Dialogue fraction', dialogue_fracs),
                    ('Entropy', entropies),
                    ('Sentence length', sent_lens),
                    ('Position in book', pos_fracs)]:
    corrs = [spearmanr(latents[:,ki], data)[0] for ki in range(K)]
    print(f'{name:<25}', ' '.join([f'{c:>8.3f}' for c in corrs]))

In [ ]:
# Test: within Tenterhooks-like books, does the oscillation track dialogue?
# Find the book with highest trajectory variance (most oscillation)

traj_variances = []
for bi, (s, e) in enumerate(story_ranges):
    traj = umap_emb[s:e]
    var = np.var(traj, axis=0).sum()
    title = books[passage_meta[s]['book']]['title']
    traj_variances.append((bi, var, title))

traj_variances.sort(key=lambda x: -x[1])
print('Most oscillating books:')
for bi, var, title in traj_variances[:5]:
    s, e = story_ranges[bi]
    df = dialogue_fracs[s:e]
    print(f'  {title[:40]}: traj_var={var:.1f}, dialogue_frac={df.mean():.3f}±{df.std():.3f}')

# For the most oscillating book, plot trajectory coloured by dialogue fraction
top_bi = traj_variances[0][0]
s, e = story_ranges[top_bi]
traj = umap_emb[s:e]
df_vals = dialogue_fracs[s:e]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))

a1.scatter(umap_emb[:,0], umap_emb[:,1], c='lightgrey', s=1, alpha=0.05)
a1.plot(traj[:,0], traj[:,1], '-', color='grey', alpha=0.3, linewidth=0.5)
sc = a1.scatter(traj[:,0], traj[:,1], c=df_vals, cmap='YlOrRd', s=30, alpha=0.8, zorder=5)
plt.colorbar(sc, ax=a1, shrink=0.7, label='Dialogue fraction')
a1.set_title(f'{traj_variances[0][2][:40]}\nTrajectory Coloured by Dialogue', fontsize=11)
a1.set_aspect('equal')

# Timeline: dialogue fraction and UMAP-x over passage index
x = np.linspace(0, 1, len(traj))
a2.plot(x, (traj[:,0] - traj[:,0].mean()) / traj[:,0].std(), 'b-', alpha=0.7, label='UMAP-x (normalised)')
a2.plot(x, (df_vals - df_vals.mean()) / max(df_vals.std(), 1e-8), 'r-', alpha=0.7, label='Dialogue frac (normalised)')
a2.set_xlabel('Position in book'); a2.set_ylabel('Normalised value')
a2.set_title('Does Trajectory Position Track Dialogue?', fontsize=11)
a2.legend(); a2.grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig(f'{SAVE}/dialogue_test.png', dpi=150); plt.show()

## 7. Mean Profiles + Archetypes

In [ ]:
from scipy.interpolate import interp1d
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

N_RESAMPLE = 50
all_resampled = []
shape_story_map = []

for si in range(n_books):
    s, e = story_ranges[si]
    z = latents[s:e]
    if len(z) < 4: continue
    x_orig = np.linspace(0, 1, len(z))
    x_new = np.linspace(0, 1, N_RESAMPLE)
    resampled = np.zeros((N_RESAMPLE, K))
    for ki in range(K):
        resampled[:, ki] = interp1d(x_orig, z[:, ki], kind='linear')(x_new)
    all_resampled.append(resampled)
    shape_story_map.append(si)

all_resampled = np.stack(all_resampled)
shape_story_map = np.array(shape_story_map)
shape_vecs = all_resampled.reshape(len(all_resampled), -1)

sil_scores = []
for nc in range(2, min(8, n_books)):
    km = KMeans(n_clusters=nc, random_state=42, n_init=10)
    cl = km.fit_predict(shape_vecs)
    sil = silhouette_score(shape_vecs, cl)
    sil_scores.append((nc, sil))
    print(f'k={nc}: silhouette={sil:.4f}')

best_nc = max(sil_scores, key=lambda x: x[1])[0]
km = KMeans(n_clusters=best_nc, random_state=42, n_init=10)
arch_labels = km.fit_predict(shape_vecs)

print(f'\nBest: {best_nc} archetypes')
for ai in range(best_nc):
    members = np.where(arch_labels == ai)[0]
    titles = [books[passage_meta[story_ranges[shape_story_map[m]][0]]['book']]['title'][:35] for m in members]
    print(f'  Type {ai+1}: {titles}')

## 8. Export JSON for Interactive Visualisation

In [ ]:
# Build export structure
export_data = {
    'metadata': {
        'model': MODEL_NAME,
        'layer': LAYER,
        'window': WINDOW,
        'ae_k': K,
        'n_books': n_books,
        'n_passages': len(passage_meta)
    },
    'books': []
}

for bi, (s, e) in enumerate(story_ranges):
    book_idx = passage_meta[s]['book']
    book_data = {
        'title': books[book_idx]['title'],
        'full_length': books[book_idx]['full_len'],
        'archetype': int(arch_labels[bi]) if bi < len(arch_labels) else -1,
        'passages': []
    }
    
    for pi in range(s, e):
        m = passage_meta[pi]
        book_data['passages'].append({
            'x': round(m['umap_x'], 4),
            'y': round(m['umap_y'], 4),
            'pos': round(m['pos_frac'], 3),
            'text': m['text_snippet'],
            'dialogue': round(m['dialogue_frac'], 3),
            'entropy': round(m['entropy'], 2),
            'sent_len': round(m['mean_sent_len'], 1)
        })
    
    export_data['books'].append(book_data)

export_path = f'{SAVE}/story_shapes.json'
with open(export_path, 'w') as f:
    json.dump(export_data, f)

size_mb = os.path.getsize(export_path) / 1e6
print(f'Exported to {export_path} ({size_mb:.1f} MB)')
print(f'{len(export_data["books"])} books, {sum(len(b["passages"]) for b in export_data["books"])} passages')

# Also save for download
from google.colab import files
files.download(export_path)

## 9. Summary

In [ ]:
print('='*60)
print('SHAPES OF STORIES — CURATED DEMO')
print('='*60)
print(f'Model: {MODEL_NAME} (layer {LAYER}/{N_LAYERS})')
print(f'Books: {n_books}')
for bi, (s, e) in enumerate(story_ranges):
    title = books[passage_meta[s]['book']]['title']
    n_pass = e - s
    print(f'  {title[:45]}: {n_pass} passages')
print(f'\nTotal passages: {len(passage_vecs)} (W={WINDOW})')
print(f'AE: PCA-128 → k={K}')
print(f'Overlapping chunks: stride={STRIDE}, overlap={OVERLAP}')
print(f'Archetypes: {best_nc}')
print(f'\nJSON exported to {export_path}')

---
# Part 2: Channel Spectral Decomposition

Graph Laplacian analysis on AE passage latents.
Tests whether the spectral structure of the passage similarity graph
corresponds to interpretable concept decompositions.

Uses `latents`, `passage_meta`, `story_ranges`, `books`, `umap_emb`,
and prose feature arrays from Part 1 above.

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score
from scipy.stats import spearmanr

print(f'Latents: {latents.shape}')
print(f'Passages: {len(passage_meta)}')
print(f'Books: {n_books}')

## 1. Build the Similarity Graph

k-NN graph on the AE latent vectors. Each passage is a node,
edges connect to k nearest neighbours weighted by Gaussian kernel.

In [ ]:
K_NN = 15  # number of nearest neighbours

# Build k-NN graph
print(f'Building {K_NN}-NN graph on {len(latents)} points in {latents.shape[1]}D...')
A_knn = kneighbors_graph(latents, n_neighbors=K_NN, mode='distance', include_self=False)

# Convert distances to Gaussian similarities
# sigma = median distance (adaptive bandwidth)
distances = A_knn.data
sigma = np.median(distances)
print(f'Median NN distance: {sigma:.4f}')

A_knn_sym = (A_knn + A_knn.T) / 2  # symmetrise

# Gaussian kernel on edges
W = A_knn_sym.copy()
W.data = np.exp(-W.data**2 / (2 * sigma**2))

n = W.shape[0]
print(f'Graph: {n} nodes, {W.nnz} edges')
print(f'Mean degree: {W.nnz / n:.1f}')

## 2. Graph Laplacian Eigendecomposition

Compute the normalised Laplacian L_norm = I - D^{-1/2} W D^{-1/2}
and find its smallest eigenvalues. The spectrum reveals the
natural number of clusters (spectral gap).

In [ ]:
# Degree matrix
degrees = np.array(W.sum(axis=1)).flatten()
D_inv_sqrt = csr_matrix(np.diag(1.0 / np.sqrt(np.maximum(degrees, 1e-10))))

# Normalised Laplacian: L_norm = I - D^{-1/2} W D^{-1/2}
# For eigsh, we compute D^{-1/2} W D^{-1/2} and find its largest eigenvalues
# (corresponding to smallest eigenvalues of L_norm)
L_rw = D_inv_sqrt @ W @ D_inv_sqrt

N_EIGS = 30  # how many eigenvalues to compute
print(f'Computing {N_EIGS} smallest eigenvalues of normalised Laplacian...')

# eigsh on the normalised adjacency (largest eigenvalues = smallest Laplacian eigenvalues)
eigenvalues_adj, eigenvectors_adj = eigsh(L_rw.tocsc(), k=N_EIGS, which='LM')

# Convert: Laplacian eigenvalues = 1 - adjacency eigenvalues
# Sort by Laplacian eigenvalue (ascending)
lap_eigenvalues = 1.0 - eigenvalues_adj
sort_idx = np.argsort(lap_eigenvalues)
lap_eigenvalues = lap_eigenvalues[sort_idx]
lap_eigenvectors = eigenvectors_adj[:, sort_idx]

# Unnormalise eigenvectors: φ = D^{-1/2} ψ
d_inv_sqrt = 1.0 / np.sqrt(np.maximum(degrees, 1e-10))
for i in range(N_EIGS):
    lap_eigenvectors[:, i] *= d_inv_sqrt

print(f'\nFirst {N_EIGS} Laplacian eigenvalues:')
for i in range(N_EIGS):
    print(f'  λ_{i:2d} = {lap_eigenvalues[i]:.6f}')

In [ ]:
# === THE SPECTRAL GAP ===
# Plot eigenvalue spectrum and look for gaps

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Full spectrum
ax = axes[0]
ax.plot(range(N_EIGS), lap_eigenvalues, 'bo-', markersize=6)
ax.set_xlabel('Index'); ax.set_ylabel('Eigenvalue')
ax.set_title('Laplacian Spectrum'); ax.grid(True, alpha=0.3)

# Gaps: difference between consecutive eigenvalues
ax = axes[1]
gaps = np.diff(lap_eigenvalues)
ax.bar(range(len(gaps)), gaps)
ax.set_xlabel('Between λ_i and λ_{i+1}'); ax.set_ylabel('Gap size')
ax.set_title('Spectral Gaps'); ax.grid(True, alpha=0.3)

# Zoom on first 10
ax = axes[2]
ax.plot(range(min(10, N_EIGS)), lap_eigenvalues[:10], 'bo-', markersize=8)
ax.set_xlabel('Index'); ax.set_ylabel('Eigenvalue')
ax.set_title('First 10 Eigenvalues (Zoom)'); ax.grid(True, alpha=0.3)

plt.suptitle('Channel Spectrum: Where Are the Concept Boundaries?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE}/spectral_gap.png', dpi=150, bbox_inches='tight'); plt.show()

# Identify largest gaps
top_gaps = np.argsort(-gaps)[:5]
print('Largest spectral gaps:')
for g in sorted(top_gaps):
    print(f'  Between λ_{g} ({lap_eigenvalues[g]:.4f}) and λ_{g+1} ({lap_eigenvalues[g+1]:.4f}): gap = {gaps[g]:.4f}')

## 3. Spectral Clusters

Use the eigenvectors below the first spectral gap as features
for clustering. Compare against UMAP-based clustering and
prose mode heuristics.

In [ ]:
# Try spectral clustering at different numbers of clusters
# Use the Laplacian eigenvectors directly as features for KMeans

results = []
for nc in range(2, 12):
    # Use first nc eigenvectors (skip eigvec 0 which is constant)
    features = lap_eigenvectors[:, 1:nc+1]
    km = KMeans(n_clusters=nc, random_state=42, n_init=10)
    labels = km.fit_predict(features)
    
    sil = silhouette_score(latents, labels)
    
    # Correlation with prose features
    book_ids = np.array([m['book'] for m in passage_meta])
    ari_books = adjusted_rand_score(book_ids, labels)
    
    results.append({
        'nc': nc, 'sil': sil, 'ari_books': ari_books, 'labels': labels
    })
    print(f'nc={nc:2d}: silhouette={sil:.4f}, ARI_books={ari_books:.4f}')

res_df = pd.DataFrame(results)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.plot(res_df['nc'], res_df['sil'], 'bo-', label='Silhouette')
a1.set_xlabel('Number of spectral clusters'); a1.set_ylabel('Silhouette')
a1.set_title('Cluster Quality'); a1.legend(); a1.grid(True, alpha=0.3)

a2.plot(res_df['nc'], res_df['ari_books'], 'rs-', label='ARI vs book identity')
a2.set_xlabel('Number of spectral clusters'); a2.set_ylabel('ARI')
a2.set_title('Do Clusters Track Books or Prose Modes?')
a2.legend(); a2.grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig(f'{SAVE}/spectral_cluster_quality.png', dpi=150); plt.show()

In [ ]:
# Select a clustering and characterise it
# Choose nc where silhouette is high but ARI_books is moderate
# (high ARI_books = just recovering book identity, not prose modes)

# Heuristic: pick nc with best ratio of silhouette to ARI_books
res_df['ratio'] = res_df['sil'] / (res_df['ari_books'] + 0.01)
best_idx = res_df['ratio'].idxmax()
best_nc = res_df.loc[best_idx, 'nc']
print(f'Selected nc={best_nc} (sil={res_df.loc[best_idx, "sil"]:.4f}, ari_books={res_df.loc[best_idx, "ari_books"]:.4f})')

# Also show what the spectral gap suggests
largest_gap_idx = np.argmax(gaps[:10]) + 1  # +1 because gap after λ_i suggests i+1 clusters
print(f'Spectral gap suggests nc={largest_gap_idx}')

# Use spectral gap suggestion if reasonable, otherwise use heuristic
NC = largest_gap_idx if 2 <= largest_gap_idx <= 10 else int(best_nc)
print(f'Using nc={NC}')

spec_features = lap_eigenvectors[:, 1:NC+1]
km_spec = KMeans(n_clusters=NC, random_state=42, n_init=10)
spec_labels = km_spec.fit_predict(spec_features)

In [ ]:
# Characterise spectral clusters by prose features
cluster_stats = []
for ci in range(NC):
    mask = spec_labels == ci
    cluster_stats.append({
        'cluster': ci,
        'n': mask.sum(),
        'dialogue_mean': dialogue_fracs[mask].mean(),
        'dialogue_std': dialogue_fracs[mask].std(),
        'entropy_mean': entropies[mask].mean(),
        'entropy_std': entropies[mask].std(),
        'sent_len_mean': sent_lens[mask].mean(),
        'sent_len_std': sent_lens[mask].std(),
        'n_books': len(set(np.array([m['book'] for m in passage_meta])[mask])),
    })

cs_df = pd.DataFrame(cluster_stats)
print(cs_df.to_string(index=False))

In [ ]:
# Visualise spectral clusters on the UMAP
spec_colors = plt.cm.Set1(np.linspace(0, 0.9, NC))

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Spectral clusters
ax = axes[0]
for ci in range(NC):
    mask = spec_labels == ci
    ax.scatter(umap_emb[mask, 0], umap_emb[mask, 1], c=[spec_colors[ci]],
               s=3, alpha=0.4, label=f'C{ci} ({mask.sum()})')
ax.legend(fontsize=8, markerscale=4)
ax.set_title(f'Spectral Clusters (nc={NC})', fontsize=12); ax.set_aspect('equal')

# Dialogue fraction for comparison
ax = axes[1]
sc = ax.scatter(umap_emb[:,0], umap_emb[:,1], c=dialogue_fracs, cmap='YlOrRd', s=3, alpha=0.4)
plt.colorbar(sc, ax=ax, shrink=0.7)
ax.set_title('Dialogue Fraction', fontsize=12); ax.set_aspect('equal')

# Entropy for comparison
ax = axes[2]
sc = ax.scatter(umap_emb[:,0], umap_emb[:,1], c=entropies, cmap='plasma', s=3, alpha=0.4)
plt.colorbar(sc, ax=ax, shrink=0.7)
ax.set_title('Entropy', fontsize=12); ax.set_aspect('equal')

plt.suptitle('Spectral Clusters vs Prose Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE}/spectral_clusters_umap.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# How much do spectral clusters capture prose mode vs book identity?
# Compare: spectral clusters vs KMeans on raw latent vs KMeans on UMAP

book_ids = np.array([m['book'] for m in passage_meta])

# Binarise dialogue: >0.2 = "dialogue passage"
dialogue_binary = (dialogue_fracs > 0.2).astype(int)

# KMeans on raw latent
km_latent = KMeans(n_clusters=NC, random_state=42, n_init=10).fit_predict(latents)

# KMeans on UMAP
km_umap = KMeans(n_clusters=NC, random_state=42, n_init=10).fit_predict(umap_emb)

print(f'ARI with book identity (higher = more book-fingerprinting):')
print(f'  Spectral:     {adjusted_rand_score(book_ids, spec_labels):.4f}')
print(f'  KMeans latent: {adjusted_rand_score(book_ids, km_latent):.4f}')
print(f'  KMeans UMAP:   {adjusted_rand_score(book_ids, km_umap):.4f}')
print()
print(f'ARI with dialogue binary (higher = more prose-mode):')
print(f'  Spectral:     {adjusted_rand_score(dialogue_binary, spec_labels):.4f}')
print(f'  KMeans latent: {adjusted_rand_score(dialogue_binary, km_latent):.4f}')
print(f'  KMeans UMAP:   {adjusted_rand_score(dialogue_binary, km_umap):.4f}')

## 4. Eigenvector Analysis

The Laplacian eigenvectors are functions on the passage graph.
Each eigenvector assigns a value to every passage. What do
these values correlate with?

In [ ]:
# Correlate each eigenvector with prose features
n_eig_show = min(12, N_EIGS)

print(f'Spearman correlations: eigenvector values vs prose features')
print(f'{"Eigvec":<10}', f'{"λ":>8}', f'{"Dialogue":>10}', f'{"Entropy":>10}',
      f'{"SentLen":>10}', f'{"Position":>10}', f'{"BookID":>10}')
print('-' * 68)

eig_corrs = []
for ei in range(n_eig_show):
    ev = lap_eigenvectors[:, ei]
    r_dial, _ = spearmanr(ev, dialogue_fracs)
    r_ent, _ = spearmanr(ev, entropies)
    r_sent, _ = spearmanr(ev, sent_lens)
    r_pos, _ = spearmanr(ev, pos_fracs)
    # For book ID: compute ratio of between-book to within-book variance
    book_means = np.array([ev[book_ids == bi].mean() for bi in range(n_books)])
    between_var = np.var(book_means)
    within_var = np.mean([np.var(ev[book_ids == bi]) for bi in range(n_books)])
    book_ratio = between_var / (within_var + 1e-10)
    
    eig_corrs.append({
        'eigvec': ei, 'lambda': lap_eigenvalues[ei],
        'dialogue': r_dial, 'entropy': r_ent,
        'sent_len': r_sent, 'position': r_pos,
        'book_var_ratio': book_ratio
    })
    print(f'φ_{ei:<7d}', f'{lap_eigenvalues[ei]:>8.4f}',
          f'{r_dial:>10.3f}', f'{r_ent:>10.3f}',
          f'{r_sent:>10.3f}', f'{r_pos:>10.3f}',
          f'{book_ratio:>10.2f}')

In [ ]:
# Visualise the first several eigenvectors on the UMAP
n_show = min(8, N_EIGS - 1)
fig, axes = plt.subplots(2, n_show//2, figsize=(5*(n_show//2), 10))

for i, ax in enumerate(axes.flat[:n_show]):
    ei = i + 1  # skip eigenvector 0 (constant)
    ev = lap_eigenvectors[:, ei]
    sc = ax.scatter(umap_emb[:,0], umap_emb[:,1], c=ev, cmap='RdBu_r', s=3, alpha=0.4,
                    vmin=np.percentile(ev, 2), vmax=np.percentile(ev, 98))
    plt.colorbar(sc, ax=ax, shrink=0.7)
    ax.set_title(f'φ_{ei} (λ={lap_eigenvalues[ei]:.4f})', fontsize=10)
    ax.set_aspect('equal')

plt.suptitle('Laplacian Eigenvectors on UMAP — The Spectral Basis of Narrative Space', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE}/eigenvectors_umap.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. Trajectory Decomposition into Spectral Components

Project each book's trajectory onto the Laplacian eigenvectors.
The projection onto φ_k gives the story's "concept-k trace" —
how much that story engages with spectral mode k at each point.

In [ ]:
# Pick 6 books to trace
show_book_idxs = np.linspace(0, n_books-1, min(6, n_books), dtype=int)
show_book_colors = plt.cm.tab10(np.linspace(0, 1, len(show_book_idxs)))

n_eig_trace = min(6, N_EIGS - 1)  # eigenvectors to project onto

fig, axes = plt.subplots(n_eig_trace, 1, figsize=(14, 3*n_eig_trace), sharex=True)
if n_eig_trace == 1: axes = [axes]

for ki, ax in enumerate(axes):
    ei = ki + 1  # skip eigvec 0
    for ci, bi in enumerate(show_book_idxs):
        s, e = story_ranges[bi]
        ev_trace = lap_eigenvectors[s:e, ei]
        x = np.linspace(0, 1, len(ev_trace))
        title = books[passage_meta[s]['book']]['title'][:25]
        ax.plot(x, ev_trace, '-', color=show_book_colors[ci], alpha=0.7, linewidth=1.5,
                label=title if ki == 0 else None)
    ax.set_ylabel(f'φ_{ei}', fontsize=11)
    ax.grid(True, alpha=0.3)
    if ki == 0:
        ax.legend(fontsize=7, ncol=3, loc='upper right')

axes[-1].set_xlabel('Position in Book')
plt.suptitle('Spectral Concept Traces — Book Trajectories Projected onto Eigenvectors', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{SAVE}/spectral_traces.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# For each eigenvector, show the most positive and most negative passages
# This reveals what the eigenvector is "about"

for ei in range(1, min(5, N_EIGS)):
    ev = lap_eigenvectors[:, ei]
    top_pos = np.argsort(-ev)[:3]
    top_neg = np.argsort(ev)[:3]
    
    print(f'\n=== Eigenvector φ_{ei} (λ={lap_eigenvalues[ei]:.4f}) ===')
    print(f'Most POSITIVE passages:')
    for idx in top_pos:
        m = passage_meta[idx]
        print(f'  [{m["title"][:25]}] dialogue={m["dialogue_frac"]:.2f} ent={m["entropy"]:.1f}')
        print(f'    "{m["text_snippet"][:120]}"')
    print(f'Most NEGATIVE passages:')
    for idx in top_neg:
        m = passage_meta[idx]
        print(f'  [{m["title"][:25]}] dialogue={m["dialogue_frac"]:.2f} ent={m["entropy"]:.1f}')
        print(f'    "{m["text_snippet"][:120]}"')

## 6. Spectral Frequency Decomposition of Trajectories

Each book's trajectory can be decomposed by spectral frequency.
Low-frequency eigenvectors = coarse structure (genre, register).
High-frequency eigenvectors = fine structure (local mode shifts).

Reconstruct trajectories using only low-freq or high-freq components
to separate arc from texture.

In [ ]:
# For one book, show reconstruction at different spectral truncations
# Pick a book with interesting trajectory (high variance)

traj_vars = []
for bi, (s, e) in enumerate(story_ranges):
    var = np.var(lap_eigenvectors[s:e, 1:8], axis=0).sum()
    traj_vars.append(var)
demo_book = np.argmax(traj_vars)
s, e = story_ranges[demo_book]
demo_title = books[passage_meta[s]['book']]['title'][:35]
print(f'Demo book: {demo_title}')

# Full spectral trajectory
full_spectral = lap_eigenvectors[s:e, 1:N_EIGS]  # (n_passages, N_EIGS-1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

truncations = [
    (1, 2, 'φ₁-φ₂ (coarsest)'),
    (1, 4, 'φ₁-φ₄ (coarse)'),
    (1, 8, 'φ₁-φ₈ (medium)'),
    (1, min(16, N_EIGS), f'φ₁-φ_{min(16, N_EIGS)-1} (fine)'),
]

for ax, (lo, hi, label) in zip(axes.flat, truncations):
    traj = lap_eigenvectors[s:e, lo:hi]
    # Project to 2D via PCA on these components
    from sklearn.decomposition import PCA
    if traj.shape[1] >= 2:
        traj_2d = PCA(n_components=2, random_state=42).fit_transform(traj)
    else:
        traj_2d = np.column_stack([traj[:, 0], np.zeros(len(traj))])
    
    pos = np.linspace(0, 1, len(traj_2d))
    ax.plot(traj_2d[:,0], traj_2d[:,1], '-', color='grey', alpha=0.3, linewidth=0.5)
    sc = ax.scatter(traj_2d[:,0], traj_2d[:,1], c=pos, cmap='coolwarm', s=20, alpha=0.7)
    ax.scatter(traj_2d[0,0], traj_2d[0,1], c='blue', s=80, marker='o', edgecolors='black', linewidths=1.5, zorder=10)
    ax.scatter(traj_2d[-1,0], traj_2d[-1,1], c='red', s=80, marker='s', edgecolors='black', linewidths=1.5, zorder=10)
    ax.set_title(f'{label}', fontsize=11)
    ax.set_aspect('equal')

plt.suptitle(f'Spectral Frequency Decomposition: {demo_title}\nBlue=start, Red=end', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE}/spectral_frequency.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Cross-Book Mode Sharing

The key test: do passages from different books that are in the
same spectral cluster actually share prose character? If yes,
these are reusable channel components (candidate split idempotents).
If no, it's just book fingerprinting.

In [ ]:
# For each spectral cluster, show which books contribute and
# what the passages look like

book_ids = np.array([m['book'] for m in passage_meta])

for ci in range(NC):
    mask = spec_labels == ci
    n_total = mask.sum()
    
    # Which books contribute?
    book_counts = {}
    for bi in book_ids[mask]:
        title = books[bi]['title'][:30]
        book_counts[title] = book_counts.get(title, 0) + 1
    
    # Sort by count
    sorted_books = sorted(book_counts.items(), key=lambda x: -x[1])
    n_books_in = len(sorted_books)
    
    # Prose stats
    d_mean = dialogue_fracs[mask].mean()
    e_mean = entropies[mask].mean()
    s_mean = sent_lens[mask].mean()
    
    print(f'=== Cluster {ci}: {n_total} passages from {n_books_in} books ===')
    print(f'    Dialogue={d_mean:.3f}, Entropy={e_mean:.2f}, SentLen={s_mean:.1f}')
    print(f'    Top books: {sorted_books[:5]}')
    
    # Sample passages from different books in this cluster
    idxs_in_cluster = np.where(mask)[0]
    # Pick from different books
    shown_books = set()
    samples = []
    for idx in idxs_in_cluster:
        bi = book_ids[idx]
        if bi not in shown_books:
            samples.append(idx)
            shown_books.add(bi)
        if len(samples) >= 3:
            break
    
    for idx in samples:
        m = passage_meta[idx]
        print(f'    [{m["title"][:25]}] "{m["text_snippet"][:100]}"')
    print()

In [ ]:
# Quantitative cross-book sharing test
# For each cluster: what fraction of books contribute at least 5% of their passages?
# High sharing = reusable mode. Low sharing = book fingerprint.

print('Cross-book sharing by spectral cluster:')
print(f'{"Cluster":>8} {"N_pass":>8} {"N_books":>8} {"Gini":>8} {"Interpretation":>20}')
print('-' * 56)

for ci in range(NC):
    mask = spec_labels == ci
    n_total = mask.sum()
    
    # Count per book
    per_book = np.array([np.sum((book_ids == bi) & mask) for bi in range(n_books)])
    per_book_frac = per_book / np.maximum(np.bincount(book_ids, minlength=n_books), 1)
    
    # Books with >5% of their passages in this cluster
    contributing = np.sum(per_book_frac > 0.05)
    
    # Gini coefficient: 0 = perfectly spread, 1 = all from one book
    sorted_counts = np.sort(per_book[per_book > 0])
    n_nonzero = len(sorted_counts)
    if n_nonzero > 0:
        index = np.arange(1, n_nonzero + 1)
        gini = (2 * np.sum(index * sorted_counts) / (n_nonzero * sorted_counts.sum())) - (n_nonzero + 1) / n_nonzero
    else:
        gini = 0
    
    interp = 'SHARED MODE' if contributing > n_books * 0.4 else 'BOOK-SPECIFIC' if contributing < 3 else 'PARTIAL'
    print(f'{ci:>8} {n_total:>8} {contributing:>8} {gini:>8.3f} {interp:>20}')

## 8. Summary

In [ ]:
print('='*60)
print('CHANNEL SPECTRAL DECOMPOSITION — SUMMARY')
print('='*60)
print(f'\nGraph: {n} nodes, {K_NN}-NN, σ={sigma:.4f}')
print(f'\nLaplacian spectrum (first 10):')
for i in range(min(10, N_EIGS)):
    print(f'  λ_{i} = {lap_eigenvalues[i]:.6f}')

print(f'\nLargest spectral gap: between λ_{np.argmax(gaps[:10])} and λ_{np.argmax(gaps[:10])+1}')
print(f'  gap = {gaps[np.argmax(gaps[:10])]:.4f}')
print(f'\nSpectral clusters: {NC}')
print(f'  Silhouette: {silhouette_score(latents, spec_labels):.4f}')
print(f'  ARI vs book identity: {adjusted_rand_score(book_ids, spec_labels):.4f}')
print(f'  ARI vs dialogue binary: {adjusted_rand_score(dialogue_binary, spec_labels):.4f}')

print(f'\nKey eigenvector correlations:')
for ec in eig_corrs[1:5]:  # skip φ_0
    best_feat = max(['dialogue', 'entropy', 'sent_len'], key=lambda f: abs(ec[f]))
    print(f'  φ_{ec["eigvec"]}: strongest corr with {best_feat} (ρ={ec[best_feat]:.3f}), book_var_ratio={ec["book_var_ratio"]:.2f}')

print(f'\nInterpretation:')
print(f'  If book_var_ratio >> 1: eigenvector tracks book identity (fingerprinting)')
print(f'  If book_var_ratio ≈ 1 and |corr| > 0.3: eigenvector tracks prose mode (channel component)')
print(f'  The spectral gap tells us how many natural concept boundaries exist.')